This notebook generates .trec files of hybrid retrieval using the .trec files of teh sparse and dense retrievers


In [1]:
# ==========================================
# 📘 build_hybrid_runs.ipynb
# ==========================================
import os
import numpy as np
from collections import defaultdict

# -------------------------------
# Configuration
# -------------------------------
RUNS_DIR = "./runs"
DATASET_NAME = "beir/arguana"  # change per dataset
TOP_K = 100
NORMALIZATIONS = ["minmax", "zscore", "rank"]
ALPHAS = [round(x, 2) for x in np.linspace(0.0, 1.0, 11)]

os.makedirs(RUNS_DIR, exist_ok=True)

# -------------------------------
# Load BM25 & Dense Runs
# -------------------------------
def read_trec(path):
    per_q = defaultdict(list)
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 6:
                continue
            qid, _, docid, _rank, score, _sys = parts[:6]
            per_q[qid].append((docid, float(score)))
    for qid in per_q:
        per_q[qid].sort(key=lambda x: x[1], reverse=True)
    return per_q

def write_trec(path, records, system_name="hybrid"):
    per_q = defaultdict(list)
    for qid, docid, score in records:
        per_q[qid].append((docid, float(score)))
    with open(path, "w", encoding="utf-8") as f:
        for qid, pairs in per_q.items():
            pairs.sort(key=lambda x: x[1], reverse=True)
            for rank, (docid, score) in enumerate(pairs[:TOP_K], start=1):
                f.write(f"{qid} Q0 {docid} {rank} {score:.6f} {system_name}\n")

bm25_path = os.path.join(RUNS_DIR, f"run_bm25_{DATASET_NAME.replace('/', '_')}.trec")
dense_path = os.path.join(RUNS_DIR, f"run_dense_{DATASET_NAME.replace('/', '_')}.trec")

assert os.path.exists(bm25_path), f"BM25 run missing: {bm25_path}"
assert os.path.exists(dense_path), f"Dense run missing: {dense_path}"

bm25 = read_trec(bm25_path)
dense = read_trec(dense_path)
print(f"✅ Loaded {len(bm25)} BM25 queries and {len(dense)} Dense queries.")


# -------------------------------
# Normalization helpers
# -------------------------------
import numpy as np

def minmax_norm(scores):
    if not scores: return {}
    vals = np.array([s for _, s in scores])
    lo, hi = vals.min(), vals.max()
    if hi <= lo: return {doc: 0.0 for doc,_ in scores}
    return {doc: (s - lo) / (hi - lo) for doc, s in scores}

def zscore_norm(scores):
    if not scores: return {}
    vals = np.array([s for _, s in scores])
    mu, sigma = vals.mean(), vals.std() + 1e-9
    return {doc: (s - mu) / sigma for doc, s in scores}

def rank_norm(scores):
    n = len(scores)
    if n == 0: return {}
    sorted_scores = sorted(scores, key=lambda x: x[1], reverse=True)
    return {doc: (n - r) / max(1, n - 1) for r, (doc, _) in enumerate(sorted_scores, start=1)}


# -------------------------------
# Fusion logic (exact alpha method)
# -------------------------------
def fuse_per_query(qid, bm25, dense, norm="minmax", alpha=0.5, topk=TOP_K):
    b = bm25.get(qid, [])
    d = dense.get(qid, [])
    if norm == "minmax":
        bn, dn = minmax_norm(b), minmax_norm(d)
    elif norm == "zscore":
        bn, dn = zscore_norm(b), zscore_norm(d)
    elif norm == "rank":
        bn, dn = rank_norm(b), rank_norm(d)
    else:
        raise ValueError(f"Unknown norm: {norm}")

    cand = set([doc for doc,_ in b]) | set([doc for doc,_ in d])
    fused = []
    for doc in cand:
        sb, sd = bn.get(doc, 0.0), dn.get(doc, 0.0)
        sc = alpha * sb + (1 - alpha) * sd
        fused.append((doc, sc))
    fused.sort(key=lambda x: x[1], reverse=True)
    return fused[:topk]


def build_hybrid_run(bm25, dense, norm, alpha, topk=TOP_K):
    recs = []
    qids = sorted(set(list(bm25.keys()) + list(dense.keys())))
    for qid in qids:
        fused = fuse_per_query(qid, bm25, dense, norm=norm, alpha=alpha, topk=topk)
        for rank, (doc, score) in enumerate(fused, start=1):
            recs.append((qid, doc, float(score)))
    return recs


# -------------------------------
# Build all hybrid runs
# -------------------------------
for norm in NORMALIZATIONS:
    for alpha in ALPHAS:
        run_name = f"run_hybrid_{norm}_a{alpha:.2f}_{DATASET_NAME.replace('/', '_')}.trec"
        out_path = os.path.join(RUNS_DIR, run_name)
        if os.path.exists(out_path):
            print(f"⏭️ Skipping existing: {run_name}")
            continue
        print(f"⚙️ Building hybrid (norm={norm}, alpha={alpha:.2f})")
        recs = build_hybrid_run(bm25, dense, norm=norm, alpha=alpha, topk=TOP_K)
        write_trec(out_path, recs, system_name=f"hybrid_{norm}_{alpha:.2f}")

print("\n✅ All hybrid runs saved in:", RUNS_DIR)


✅ Loaded 1406 BM25 queries and 1406 Dense queries.
⚙️ Building hybrid (norm=minmax, alpha=0.00)
⚙️ Building hybrid (norm=minmax, alpha=0.10)
⚙️ Building hybrid (norm=minmax, alpha=0.20)
⚙️ Building hybrid (norm=minmax, alpha=0.30)
⚙️ Building hybrid (norm=minmax, alpha=0.40)
⚙️ Building hybrid (norm=minmax, alpha=0.50)
⚙️ Building hybrid (norm=minmax, alpha=0.60)
⚙️ Building hybrid (norm=minmax, alpha=0.70)
⚙️ Building hybrid (norm=minmax, alpha=0.80)
⚙️ Building hybrid (norm=minmax, alpha=0.90)
⚙️ Building hybrid (norm=minmax, alpha=1.00)
⚙️ Building hybrid (norm=zscore, alpha=0.00)
⚙️ Building hybrid (norm=zscore, alpha=0.10)
⚙️ Building hybrid (norm=zscore, alpha=0.20)
⚙️ Building hybrid (norm=zscore, alpha=0.30)
⚙️ Building hybrid (norm=zscore, alpha=0.40)
⚙️ Building hybrid (norm=zscore, alpha=0.50)
⚙️ Building hybrid (norm=zscore, alpha=0.60)
⚙️ Building hybrid (norm=zscore, alpha=0.70)
⚙️ Building hybrid (norm=zscore, alpha=0.80)
⚙️ Building hybrid (norm=zscore, alpha=0.90)
⚙️ B